# From Pipeline to Agent

In the **last class** we built an NL-to-SQL pipeline. It worked, but it could only do one thing: write SQL and run it. Today we turn it into an **agent**.

## What is the difference?

A **pipeline** is a fixed sequence of steps. We had six of them, in this order, every time:

```
question  ->  generate SQL  ->  validate  ->  execute  ->  summarize  ->  answer
```

An **agent** does not have a fixed sequence. Instead, it has **tools**, and on every turn it decides what to do next:

```
                +--------> tool A (e.g. SQL)
                |
question -> LLM +--------> tool B (e.g. lookup)
                |              |
                |              v
                +<------- observation
                |
                +-> answer (when it has enough)
```

This loop is called **ReAct** (Reason + Act). On every iteration the LLM either:
- emits a tool call (the action), then we run the tool and feed the result back, OR
- emits a final answer (and we stop).

## Why we need this

Look at the questions our pipeline could **not** answer last class:

- "What is the thrust of the rocket we use for crewed missions?"
- "Which of our rockets is reusable?"
- "Compare Falcon Heavy and Starship: payload capacity and our missions on each."

Our SQL database does not have rocket specs (thrust, height, reusability). Those facts live in a different file (`vehicle_specs.json`). So a single SQL query is not enough. The agent will pick which tool to call.


## Setup


In [31]:
# uv add reads pyproject.toml in this folder.
# If you don't have uv: curl -LsSf https://astral.sh/uv/install.sh | sh
!uv add langchain-openai langchain-core pandas

Resolved 142 packages in 4ms
Checked 130 packages in 45ms


In [32]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [33]:
import sqlite3
from pathlib import Path
import json
import random
import pandas as pd
import requests
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

DB = Path("../spacex_launches.db")

SCHEMA = Path("../schema.md").read_text()
SPECS = json.loads(Path("../vehicle_specs.json").read_text())

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. The two data sources

The agent will have access to:

1. **`spacex_launches.db`** - the same SQLite database from last class. One table, `launches`.
2. **`vehicle_specs.json`** - a JSON file of rocket specs (thrust, height, reusability, etc.). Same data shape we'd hit if we were calling a microservice.

Two completely different storage formats. Two different access patterns. The agent has to know which one to ask.


In [34]:
# launches table - structured, query with SQL
con = sqlite3.connect(DB)
df_launches = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df_launches.head(5)

,launch_id,mission_name,vehicle,payload_type,payload_kg,destination,success,launch_date,customer
0,1,Falcon 1 Flight 1,Falcon 1,Test,20,LEO,0,2006-03-24,DARPA
1,2,Falcon 1 Flight 4,Falcon 1,Test,165,LEO,1,2008-09-28,SpaceX
2,3,COTS Demo Flight 2,Falcon 9,Cargo,525,ISS,1,2012-05-22,NASA
3,4,CASSIOPE,Falcon 9,Satellite,500,LEO,1,2013-09-29,MDA
4,5,DSCOVR,Falcon 9,Satellite,570,L1,1,2015-02-11,NOAA


In [35]:
# vehicle specs - JSON file, lookup by key
print(json.dumps(SPECS, indent=2)[:900])
print("...")

{
  "Falcon 1": {
    "first_flight": "2006-03-24",
    "height_m": 21,
    "diameter_m": 1.7,
    "mass_kg": 30000,
    "thrust_kn": 454,
    "payload_to_leo_kg": 670,
    "stages": 2,
    "reusable": false,
    "active": false,
    "manufacturer": "SpaceX",
    "notes": "First privately-developed liquid rocket to reach orbit. Retired in 2009.",
    "crew_capacity": 5
  },
  "Falcon 9": {
    "first_flight": "2010-06-04",
    "height_m": 70,
    "diameter_m": 3.7,
    "mass_kg": 549054,
    "thrust_kn": 7607,
    "payload_to_leo_kg": 22800,
    "stages": 2,
    "reusable": true,
    "active": true,
    "manufacturer": "SpaceX",
    "notes": "Workhorse rocket. First stage is reusable - lands on a droneship or back at the launch site.",
    "crew_capacity": 10
  },
  "Falcon Heavy": {
    "first_flight": "2018-02-06",
    "height_m": 70,
    "diameter_m": 12.2,
    "mass_kg": 1420788,
   
...


## 2. A question the pipeline cannot answer

> *"What is the thrust of the rocket we use for crewed missions?"*

To answer this, you need to:

1. Find which vehicle is used for crewed missions. **(SQL on `launches`)**
2. Look up the thrust of that vehicle. **(JSON lookup in `vehicle_specs`)**

A pipeline always uses SQL. It can find the vehicle (Falcon 9), but it cannot then go look up its thrust. The agent will do both.


## 3. Define the tools

A **tool** in LangChain is just a Python function with a description. The LLM reads the description and decides when to call it.

The `@tool` decorator wraps the function so the LLM can see its name, arguments, and docstring. We define two:


In [36]:
FORBIDDEN = ("INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE")
 
@tool
def get_crypto_price(coin: str) -> str:
    """
    Get live cryptocurrency price in USD.
    Example coins: bitcoin, ethereum, solana
    """

    url = (
        "https://api.coingecko.com/api/v3/simple/price"
        f"?ids={coin}&vs_currencies=usd"
    )

    response = requests.get(url)
    data = response.json()

    if coin not in data:
        return f"Could not find coin: {coin}"

    price = data[coin]["usd"]

    return f"{coin} price is ${price}"

@tool
def calculate(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Examples:
    2 + 2
    (45 * 12) / 3
    100 ** 2
    """

    try:
        result = eval(expression)

        return f"Result: {result}"

    except Exception as e:
        return f"Error: {str(e)}"
    
@tool
def compare(a: str, b: str, label: str) -> str:
    """
    Compare two values based on the provided comparison label.

    The label describes:
    - what is being compared
    - how the comparison should be interpreted

    The agent should:
    - choose meaningful values for `a` and `b`
    - decide comparison context using the label
    - use this tool when synthesizing information
      from multiple tool calls instead of manually
      formatting comparisons.

    Returns a formatted comparison string.
    """

    return f"{label}: a={a}, b={b}"
    
@tool
def random_number() -> int:
    """
    Returns a random number.
    This tool is intentionally irrelevant for evaluation testing.
    """

    return random.randint(1, 1000)

@tool
def spacex_data_tool(
    query_type: str,
    query: str
) -> str:
    """
    Unified SpaceX data tool.

    query_type:
    - 'vehicle_specs' → look up rocket specifications
    - 'launch_sql'    → execute SQL against launches database
    """
    if query_type == "vehicle_specs":

        spec = SPECS.get(query)

        if spec is None:
            return (
                f"UNKNOWN VEHICLE: {query!r}. "
                f"Allowed: {list(SPECS.keys())}"
            )

        return json.dumps(spec, indent=2)
    elif query_type == "launch_sql":

        try:
            con = sqlite3.connect(DB)

            rows = con.execute(query).fetchall()

            con.close()

            return str(rows)

        except Exception as e:
            return f"SQL ERROR: {str(e)}"

    else:
        return (
            "Invalid query_type. "
            "Use 'vehicle_specs' or 'launch_sql'."
        )

tools = [
    get_crypto_price,
    random_number,
    compare,
    calculate,
    spacex_data_tool
]
TOOLS_BY_NAME = {t.name: t for t in tools}

print("Tools defined:", [t.name for t in tools])

Tools defined: ['get_crypto_price', 'random_number', 'compare', 'calculate', 'spacex_data_tool']


## 4. Bind the tools to the LLM

`bind_tools()` tells the LLM which tools exist. The LLM does **not** call the functions itself - it emits a "tool call" object describing which function to run with which arguments. We then run it ourselves in Python and feed the result back. This is what every modern agent framework does under the hood.


In [37]:
agent_llm = llm.bind_tools(tools)

# Quick check - send a simple question and inspect the response
test_response = agent_llm.invoke([
    SystemMessage(content="You answer questions using the tools provided."),
    HumanMessage(content="What is the thrust of Falcon 9?"),
])

print("LLM content:    ", test_response.content)
print("LLM tool_calls: ", test_response.tool_calls)

LLM content:     
LLM tool_calls:  [{'name': 'spacex_data_tool', 'args': {'query_type': 'vehicle_specs', 'query': 'Falcon 9'}, 'id': 'call_z6U5N2uQGk7Wc8WoyE70uqPC', 'type': 'tool_call'}]


See? The LLM did **not** answer in text. It returned a `tool_call` saying *"please call `vehicle_specs` with `vehicle='Falcon 9'`"*. We have to actually run the function and feed back the result. That is the agent's job.


## 5. The agent's system prompt

The system prompt is shorter than the pipeline's was. We do not tell the LLM how to write SQL exhaustively - we just hand it tools and tell it when to use which.

We do still include `schema.md` so the LLM knows what columns exist (so it can write good SQL inside the `query_launches` tool call) and what domain terms mean.


In [38]:
AGENT_SYSTEM = f"""You are an analyst with access to four tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.
3. `get_crypto_price(coin)` - get live cryptocurrency price in USD.
4. `get_random_joke()` - get a random programming joke.

Use the tools to gather facts. When you have enough information, answer the user
in one short paragraph. If a tool returns an error or no rows,
say so honestly.

Do not call a tool if you can answer from prior tool results. Do not guess - look up.

Schema for the SQL tool:

{SCHEMA}
"""

print(AGENT_SYSTEM[:600])
print("...")

You are an analyst with access to four tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.
3. `get_crypto_price(coin)` - get live cryptocurrency price in USD.
4. `get_random_joke()` - get a random programming joke.

Use the tools to gather facts. When you have enough information, answer the user
in one short paragraph. If a tool returns an error or no rows,
say so honestly.

Do not call a tool if you can answer from prior tool results. Do not guess - look up.

Schema for the SQL tool:

# 
...


## 6. The agent loop (ReAct, written manually)

The whole agent is one Python function with a loop. On each iteration we:

1. Send the conversation so far to the LLM.
2. If the LLM emits a tool call, we run the tool and append its result.
3. If the LLM emits text content with no tool call, that's the final answer.
4. We bail out after a max number of iterations as a safety net.

This is the **same loop** that frameworks like LangGraph, CrewAI, AutoGen wrap up. Writing it ourselves makes the abstraction concrete.


In [39]:
def run_agent(question, max_iters=8, verbose=True):
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]
    if verbose:
        print(f"USER: {question}\n")

    for step in range(1, max_iters + 1):
        response = agent_llm.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print(f"--- step {step}: final answer ---")
                print(response.content)
            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            if verbose:
                print(f"--- step {step}: action ---")
                print(f"  tool: {name}")
                print(f"  args: {args}")
            tool_fn = TOOLS_BY_NAME[name]
            result = tool_fn.invoke(args)
            if verbose:
                short = result[:240] + ("..." if len(result) > 240 else "")
                print(f"  observation: {short}\n")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Agent hit max iterations without finishing."


# Quick smoke test
_ = run_agent("What is the thrust of Falcon 9?", max_iters=3)

USER: What is the thrust of Falcon 9?

--- step 1: action ---
  tool: get_crypto_price
  args: {'coin': 'falcon 9'}
  observation: Could not find coin: falcon 9

--- step 2: action ---
  tool: get_crypto_price
  args: {'coin': 'falcon'}
  observation: falcon price is $6.126e-05

--- step 3: action ---
  tool: get_crypto_price
  args: {'coin': '9'}
  observation: Could not find coin: 9



## 7. Trace a multi-tool question end-to-end

Now the question that motivated this whole class:

> *"What is the thrust of the rocket we use for crewed missions?"*

Watch how the agent reasons. It needs to call **both** tools to answer.


In [40]:
_ = run_agent("What is the thrust of the rocket we use for crewed missions?")

USER: What is the thrust of the rocket we use for crewed missions?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT vehicle FROM launches WHERE payload_type = 'Crew' LIMIT 1"}
  observation: [('Falcon 9',)]

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon 9'}
  observation: {
  "first_flight": "2010-06-04",
  "height_m": 70,
  "diameter_m": 3.7,
  "mass_kg": 549054,
  "thrust_kn": 7607,
  "payload_to_leo_kg": 22800,
  "stages": 2,
  "reusable": true,
  "active": true,
  "manufacturer": "SpaceX",
  "notes": "Wo...

--- step 2: final answer ---
The rocket used for crewed missions is the Falcon 9, which has a thrust of 7607 kN.


**Read the trace.** The agent:

1. Called `query_launches` with `SELECT DISTINCT vehicle FROM launches WHERE payload_type = 'Crew'` to find the rocket.
2. Got back `Falcon 9`.
3. Called `vehicle_specs('Falcon 9')` to get the thrust.
4. Composed the final answer using both pieces.

This is the **whole point of an agent**: the LLM decided which tool to call, and in what order, and when it had enough to answer. We did not script it.


## 8. Try more multi-tool questions

Each of these requires a different mix of tool calls. Watch the traces.


In [41]:
_ = run_agent("Which of our rockets have maximum crew capacity?")

USER: Which of our rockets have maximum crew capacity?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon 1'}
  observation: {
  "first_flight": "2006-03-24",
  "height_m": 21,
  "diameter_m": 1.7,
  "mass_kg": 30000,
  "thrust_kn": 454,
  "payload_to_leo_kg": 670,
  "stages": 2,
  "reusable": false,
  "active": false,
  "manufacturer": "SpaceX",
  "notes": "Firs...

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon 9'}
  observation: {
  "first_flight": "2010-06-04",
  "height_m": 70,
  "diameter_m": 3.7,
  "mass_kg": 549054,
  "thrust_kn": 7607,
  "payload_to_leo_kg": 22800,
  "stages": 2,
  "reusable": true,
  "active": true,
  "manufacturer": "SpaceX",
  "notes": "Wo...

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon Heavy'}
  observation: {
  "first_flight": "2018-02-06",
  "height_m": 70,
  "diameter_m": 12.2,
  "m

In [42]:
_ = run_agent("How many missions have flown on reusable rockets?")

USER: How many missions have flown on reusable rockets?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT COUNT(*) AS n FROM launches WHERE vehicle IN ('Falcon 9', 'Falcon Heavy') AND payload_kg > 0;"}
  observation: [(12,)]

--- step 2: final answer ---
There have been 12 missions that have flown on reusable rockets.


In [43]:
_ = run_agent(
    "Compare Falcon Heavy and Starship. Which has higher thrust, "
    "and how many successful missions has each flown?"
)

USER: Compare Falcon Heavy and Starship. Which has higher thrust, and how many successful missions has each flown?

--- step 1: action ---
  tool: get_crypto_price
  args: {'coin': 'falcon heavy'}
  observation: Could not find coin: falcon heavy

--- step 1: action ---
  tool: get_crypto_price
  args: {'coin': 'starship'}
  observation: starship price is $0.01280786

--- step 2: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon Heavy'}
  observation: {
  "first_flight": "2018-02-06",
  "height_m": 70,
  "diameter_m": 12.2,
  "mass_kg": 1420788,
  "thrust_kn": 22819,
  "payload_to_leo_kg": 63800,
  "stages": 2,
  "reusable": true,
  "active": true,
  "manufacturer": "SpaceX",
  "notes": ...

--- step 2: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Starship'}
  observation: {
  "first_flight": "2021-03-03",
  "height_m": 120,
  "diameter_m": 9,
  "mass_kg": 5000000,
  "thrust_kn": 74000,
  "payload_to_leo

In [44]:
_ = run_agent(
    """
Which successful mission carried the highest payload,
and what vehicle was used?
"""
)

USER: 
Which successful mission carried the highest payload,
and what vehicle was used?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT mission_name, vehicle, payload_kg FROM launches WHERE success = 1 ORDER BY payload_kg DESC LIMIT 1;'}
  observation: [('Starlink Group 4-7', 'Falcon 9', 13620)]

--- step 2: final answer ---
The successful mission that carried the highest payload was "Starlink Group 4-7," and it used the Falcon 9 vehicle, with a payload of 13,620 kg.


In [45]:
_ = run_agent(
    """
Compare Falcon 9 and Starship successful launches.
Which has more successful missions?
""")

USER: 
Compare Falcon 9 and Starship successful launches.
Which has more successful missions?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT COUNT(*) AS n_successful FROM launches WHERE vehicle = 'Falcon 9' AND success = 1;"}
  observation: [(9,)]

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT COUNT(*) AS n_successful FROM launches WHERE vehicle = 'Starship' AND success = 1;"}
  observation: [(2,)]

--- step 2: final answer ---
Falcon 9 has 9 successful launches, while Starship has 2 successful launches. Therefore, Falcon 9 has more successful missions than Starship.


In [46]:
_ = run_agent(
    """
Which mission had the cheapest launch cost?
""")

USER: 
Which mission had the cheapest launch cost?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT mission_name, vehicle, payload_kg FROM launches ORDER BY payload_kg ASC LIMIT 1;'}
  observation: [('Starship SN10', 'Starship', 0)]

--- step 2: final answer ---
The mission with the cheapest launch cost was "Starship SN10," which had no orbital payload (0 kg).


In [47]:
_ = run_agent(
    """
Which rocket was the best?
""")

USER: 
Which rocket was the best?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT vehicle, ROUND(100.0 * SUM(1 - success) / COUNT(*), 1) AS failure_pct, COUNT(*) AS n_launches FROM launches GROUP BY vehicle ORDER BY failure_pct ASC LIMIT 1;'}
  observation: [('Falcon Heavy', 0.0, 2)]

--- step 2: final answer ---
The best rocket, based on the lowest failure rate, is the Falcon Heavy, which has a failure percentage of 0.0% across its 2 launches.


## 9. Pipeline vs agent on the same question

For SQL-only questions (like the ones from last class), the agent reduces to one tool call. Same answer, slightly more overhead. The agent's value shows up when **the question requires more than one source**.


In [48]:
# A simple SQL-only question - the agent uses one tool
_ = run_agent("How many launches were successful?", verbose=True)

USER: How many launches were successful?

--- step 1: action ---
  tool: get_crypto_price
  args: {'coin': 'bitcoin'}
  observation: bitcoin price is $81277

--- step 2: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT COUNT(*) AS n FROM launches WHERE success = 1;'}
  observation: [(14,)]

--- step 3: final answer ---
There have been 14 successful launches.


In [49]:
# A specs-only question - the agent uses one tool, the OTHER one
_ = run_agent("How tall is Starship?", verbose=True)

USER: How tall is Starship?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Starship'}
  observation: {
  "first_flight": "2021-03-03",
  "height_m": 120,
  "diameter_m": 9,
  "mass_kg": 5000000,
  "thrust_kn": 74000,
  "payload_to_leo_kg": 250000,
  "stages": 2,
  "reusable": true,
  "active": true,
  "manufacturer": "SpaceX",
  "notes": "...

--- step 2: final answer ---
Starship is 120 meters tall.


## 10. Failure modes


### Failure 1: the agent makes things up when both tools come back empty

Ask about something neither tool can answer. The agent SHOULD say it doesn't know. Watch.


In [50]:
_ = run_agent("What is the diameter of the Saturn V rocket?", max_iters=4)

USER: What is the diameter of the Saturn V rocket?

--- step 1: action ---
  tool: get_crypto_price
  args: {'coin': 'saturn-v'}
  observation: Could not find coin: saturn-v

--- step 2: action ---
  tool: get_crypto_price
  args: {'coin': 'saturn'}
  observation: Could not find coin: saturn

--- step 3: action ---
  tool: get_crypto_price
  args: {'coin': 'v'}
  observation: Could not find coin: v

--- step 4: final answer ---
I cannot provide the diameter of the Saturn V rocket as I do not have access to that specific information. You may want to look it up in a reliable source or database related to space exploration.


### Failure 2: the agent runs in circles

If we give a vague question, the agent may keep calling tools without converging. The `max_iters` safety valve catches it.


In [51]:
_ = run_agent("Tell me everything interesting about our missions.", max_iters=4)

USER: Tell me everything interesting about our missions.

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT mission_name, vehicle, payload_type, payload_kg, destination, success, launch_date, customer FROM launches ORDER BY launch_date DESC;'}
  observation: [('Europa Clipper', 'Falcon Heavy', 'Probe', 6065, 'Jupiter Transfer', 1, '2024-10-14', 'NASA'), ('Polaris Dawn', 'Falcon 9', 'Crew', 12500, 'LEO', 1, '2024-09-10', 'Shift4'), ('Starship IFT-3', 'Starship', 'Test', 0, 'Suborbital', 1, '2024...

--- step 2: final answer ---
SpaceX has conducted a variety of interesting missions, including the upcoming Europa Clipper probe launch on a Falcon Heavy rocket, scheduled for October 14, 2024, aimed at exploring Jupiter. The Polaris Dawn mission, a crewed flight on Falcon 9, is set for September 10, 2024, with a payload of 12,500 kg to Low Earth Orbit (LEO). Notable test flights include the Starship IFT-3, which successfully launched subor

### Failure 3: validator catches dangerous SQL

The same `validate` logic from last class lives inside the `query_launches` tool. The agent cannot bypass it.


In [52]:
_ = run_agent("Delete the failed launches from the database.", max_iters=3)

USER: Delete the failed launches from the database.

--- step 1: final answer ---
I cannot perform delete operations on the database. I can only run SELECT queries to retrieve information.


## 11. What this agent still cannot do

We solved the multi-source problem. We did not solve everything:

| Gap                                  | What you'd add                                                  |
|--------------------------------------|------------------------------------------------------------------|
| No memory across runs                 | Persist the messages list to disk per user/conversation.         |
| No unstructured docs (press releases) | A vector-search tool over a document store.                      |
| No real-time data                     | A web-search tool, or live API calls.                             |
| No charts                              | A `make_chart(spec)` tool that returns an image.                  |
| Confident hallucinations               | Stricter prompts + a "I'm not sure" escape hatch in the agent.   |
| Cost / latency creep                   | Tool-call caching + max-iters tuning + smaller models for retries.|

Adding tools is cheap; the hard part is choosing which tools are worth the agent's attention budget.


## 12. Try it yourself

1. Add a third tool: `compare(a, b, label)` that just returns `f"{label}: a={a}, b={b}"`. Ask a comparison question. Does the agent use it?
2. Add a fourth tool that the agent should *never* need to use (e.g. `random_number()`). Does the agent leave it alone?
3. Edit `vehicle_specs.json` and add a new key (`"crew_capacity"`). Ask a question that needs that key. Does the agent pull it out correctly?
4. Drop `temperature` from 0 to 0.7 and re-run the comparison question three times. Are the traces identical? What changed?
5. Replace `gpt-4o-mini` with `gpt-3.5-turbo` and run the multi-tool question. Does the agent still call both tools, or does it skip one?
6. Add `print(messages)` after the loop. How does the conversation grow with each step?


## Other Tools

1. compare
2. Random numbers

In [53]:
_ = run_agent("""Compare Falcon 9 and Starship successful launches.
        Which has more successful missions?""")

USER: Compare Falcon 9 and Starship successful launches.
        Which has more successful missions?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT COUNT(*) AS n_successful FROM launches WHERE vehicle = 'Falcon 9' AND success = 1;"}
  observation: [(9,)]

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': "SELECT COUNT(*) AS n_successful FROM launches WHERE vehicle = 'Starship' AND success = 1;"}
  observation: [(2,)]

--- step 2: final answer ---
Falcon 9 has 9 successful launches, while Starship has 2 successful launches. Therefore, Falcon 9 has more successful missions than Starship.


## Temperature change to 0.7 from 0

In [54]:
import trace


llm_temperature = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
agent_llm_temp= llm_temperature.bind_tools(tools)

def run_agent_temp(question, max_iters=8, verbose=True):
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]
    if verbose:
        print(f"USER: {question}\n")

    for step in range(1, max_iters + 1):
        response = agent_llm_temp.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print(f"--- step {step}: final answer ---")
                print(response.content)
            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            if verbose:
                print(f"--- step {step}: action ---")
                print(f"  tool: {name}")
                print(f"  args: {args}")
            tool_fn = TOOLS_BY_NAME[name]
            result = tool_fn.invoke(args)
            if verbose:
                short = result[:240] + ("..." if len(result) > 240 else "")
                print(f"  observation: {short}\n")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

        return {
                "answer": "Agent hit max iterations without finishing.",
                "trace": trace
            }


# Quick smoke test
question = """
Compare payload of all rockets,
show the specifications of that rocket?
"""

for i in range(3):

    print(f"\n========== RUN {i+1} ==========\n")

    result = run_agent_temp(question,
        verbose=True
    )

    print("\nTRACE:")
    print(result["trace"])
    


========== RUN 1 ==========

USER: 
Compare payload of all rockets,
show the specifications of that rocket?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT vehicle, payload_kg FROM launches ORDER BY payload_kg DESC'}
  observation: [('Falcon 9', 13620), ('Falcon 9', 12500), ('Falcon 9', 12500), ('Falcon 9', 12500), ('Falcon 9', 9600), ('Falcon Heavy', 6065), ('Falcon 9', 1952), ('Falcon Heavy', 1350), ('Falcon 9', 610), ('Falcon 9', 570), ('Falcon 9', 525), ('Falcon 9...


TRACE:
<module 'trace' from 'C:\\Python314\\Lib\\trace.py'>

========== RUN 2 ==========

USER: 
Compare payload of all rockets,
show the specifications of that rocket?


--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'launch_sql', 'query': 'SELECT vehicle, MAX(payload_kg) AS max_payload FROM launches GROUP BY vehicle'}
  observation: [('Falcon 1', 165), ('Falcon 9', 13620), ('Falcon Heavy', 6065), ('Starship', 0)]


TRACE:
<module 'trac

## Change Model From  gpt-4o-mini to gpt-3.5-turbo 

In [55]:

llm_model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
agent_llm_model= llm_model.bind_tools(tools)

def run_agent_model(question, max_iters=8, verbose=True):
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]
    if verbose:
        print(f"USER: {question}\n")

    for step in range(1, max_iters + 1):
        response = agent_llm_model.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print(f"--- step {step}: final answer ---")
                print(response.content)
            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]
            if verbose:
                print(f"--- step {step}: action ---")
                print(f"  tool: {name}")
                print(f"  args: {args}")
            tool_fn = TOOLS_BY_NAME[name]
            result = tool_fn.invoke(args)
            if verbose:
                short = result[:240] + ("..." if len(result) > 240 else "")
                print(f"  observation: {short}\n")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

            return "Agent hit max iterations without finishing."


_ = run_agent_model("What is the thrust of the rocket we use for crewed missions?")

USER: What is the thrust of the rocket we use for crewed missions?

--- step 1: action ---
  tool: spacex_data_tool
  args: {'query_type': 'vehicle_specs', 'query': 'Falcon 9'}
  observation: {
  "first_flight": "2010-06-04",
  "height_m": 70,
  "diameter_m": 3.7,
  "mass_kg": 549054,
  "thrust_kn": 7607,
  "payload_to_leo_kg": 22800,
  "stages": 2,
  "reusable": true,
  "active": true,
  "manufacturer": "SpaceX",
  "notes": "Wo...



## Add Message Log


In [60]:
import trace

llm_print = ChatOpenAI(model="gpt-4o-mini", temperature=0)
agent_llm_print = llm_print.bind_tools(tools)

def run_agent_print(question, max_iters=15, verbose=True):
    messages = [
        SystemMessage(content=AGENT_SYSTEM),
        HumanMessage(content=question),
    ]

    if verbose:
        print("=" * 80)
        print(" AGENT EXECUTION STARTED")
        print("=" * 80)
        print(f"USER QUESTION:\n{question}\n")

    for step in range(1, max_iters + 1):

        if verbose:
            print("\n" + "=" * 80)
            print(f" STEP {step} - SENDING MESSAGES TO LLM")
            print("=" * 80)

            print("\n CURRENT MESSAGE STACK:")
            for idx, msg in enumerate(messages):
                print(f"\n--- MESSAGE {idx+1} ---")
                print(f"TYPE: {type(msg).__name__}")

                try:
                    print(f"CONTENT:\n{str(msg.content)[:300]}")
                except Exception as e:
                    print(f"Could not print content: {e}")

        response = agent_llm_print.invoke(messages)

        if verbose:
            print("\n" + "-" * 80)
            print(" RAW LLM RESPONSE")
            print("-" * 80)

            print(f"TYPE: {type(response).__name__}")
            print(f"CONTENT:\n{str(response.content)[:300]}")

            print("\n TOOL CALLS:")
            print(response.tool_calls)

        messages.append(response)

        if not response.tool_calls:
            # Final answer
            if verbose:
                print("\n" + "=" * 80)
                print(f" FINAL ANSWER FOUND AT STEP {step}")
                print("=" * 80)
                print(response.content)

            return response.content

        # Run each tool the LLM asked for
        for call in response.tool_calls:
            name = call["name"]
            args = call["args"]

            if verbose:
                print("\n" + "=" * 80)
                print(f" TOOL EXECUTION - STEP {step}")
                print("=" * 80)

                print(f"TOOL NAME : {name}")
                print(f"TOOL ARGS : {args}")

            tool_fn = TOOLS_BY_NAME[name]

            if verbose:
                print("\n RUNNING TOOL...")

            result = tool_fn.invoke(args)

            if verbose:
                print("\n TOOL RESULT RECEIVED")
                print("-" * 80)

                short = result[:1000] + ("..." if len(result) > 1000 else "")
                print(short)

                print("-" * 80)
                print(f"RESULT TYPE: {type(result).__name__}")

                try:
                    print(f"RESULT LENGTH: {len(result)}")
                except:
                    pass

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"]
                )
            )

            if verbose:
                print("\n TOOL MESSAGE APPENDED TO MEMORY")

        return {
            "answer": "Agent hit max iterations without finishing.",
            "trace": trace
        }


# Quick smoke test
question = """
Which of our rockets have maximum crew capacity?
"""
result = run_agent_print(question,
        verbose=True
    )

print(result)

 AGENT EXECUTION STARTED
USER QUESTION:

Which of our rockets have maximum crew capacity?



 STEP 1 - SENDING MESSAGES TO LLM

 CURRENT MESSAGE STACK:

--- MESSAGE 1 ---
TYPE: SystemMessage
CONTENT:
You are an analyst with access to four tools:

1. `query_launches(sql)` - run a SELECT query on the SQLite launches table.
2. `vehicle_specs(vehicle)` - look up technical specs for a SpaceX rocket.
3. `get_crypto_price(coin)` - get live cryptocurrency price in USD.
4. `get_random_joke()` - get a ran

--- MESSAGE 2 ---
TYPE: HumanMessage
CONTENT:

Which of our rockets have maximum crew capacity?


--------------------------------------------------------------------------------
 RAW LLM RESPONSE
--------------------------------------------------------------------------------
TYPE: AIMessage
CONTENT:


 TOOL CALLS:
[{'name': 'spacex_data_tool', 'args': {'query_type': 'vehicle_specs', 'query': 'Falcon 1'}, 'id': 'call_x0kXKr9EgrW4N5FcBZs8KQO4', 'type': 'tool_call'}, {'name': 'spacex_data_tool'